# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [ ]:
import os
import warnings
from pprint import pprint
from typing import Any

from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

tqdm.pandas()
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

In [ ]:
import pandas as pd

In [4]:
from experiments.constants import TTS_CONFIGS
from experiments.nlp import TTS

For our research we'll use british english.

In [ ]:
LANGUAGE_CODES: tuple[str, ...] = (
    "en-GB",  # English
    "es-ES",  # Spanish
    "fr-FR",  # French
)

SAMPLES: dict[str, str] = {
    "en": (
        "Provide academic quotes that specifically talk about "
        "the relevance of the social component in the learning process.?"
    ),
    "es": (
        "Proporciona citas académicas que hablen específicamente "
        "sobre la relevancia del componente social en el proceso de aprendizaje."
    ),
    "fr": (
        "Fournissez des citations académiques qui parlent spécifiquement "
        "de la pertinence de la composante sociale dans le processus d’apprentissage"
    ),
}

tts: TTS = TTS()


def display_sample_df(row: pd.Series) -> None:
    """
    Display the audio sample along with its metadata.

    Args:
        row: A row from the DataFrame containing audio metadata and audio data.
    """
    print(f"Language: {row['language_code']}, Gender: {row['gender']}, Voice: {row['voice_name']}")
    tts.display_audio(row["audio"])

# TTS Demo

In [ ]:
TTS.display_audio(
    await tts.synthesize_text(  # type: ignore[top-level-await] # pylint: disable=await-outside-async # noqa: E501,F704
        text="Hello, this is a demo of Google Text-to-Speech.",
        language_code="en-GB",
        voice_name="en-GB-Wavenet-D",
        gender=1,  # type: ignore[arg-type]
    )
)

In [ ]:
TTS.display_audio(
    await tts.synthesize_text(  # type: ignore[top-level-await] # pylint: disable=await-outside-async # noqa: E501,F704
        text="Hello, this is a demo of Google Text-to-Speech.",
        language_code="en-GB",
        voice_name="en-GB-Chirp3-HD-Achird",
        gender=1,  # type: ignore[arg-type]
    )
)

# Voices analysis

Let's assemble all available voices into a pandas data frame.

In [8]:
voices_df: pd.DataFrame = pd.DataFrame(
    [
        {"name": voice.name, "language_codes": voice.language_codes, "gender": voice.ssml_gender.name}
        for voice in (await tts.client.list_voices()).voices  # type: ignore[top-level-await] # pylint: disable=await-outside-async # noqa: E501,F704
    ]
)
voices_df.head(3)

,name,language_codes,gender
0,Achernar,[en-US],FEMALE
1,Achird,[en-US],MALE
2,Algenib,[en-US],MALE


Variety of languages are supported. Note that there are different dialects supported withing one language, ex. for english.

In [9]:
pprint(sorted(voices_df.explode("language_codes")["language_codes"].unique()))

['af-ZA',
 'am-ET',
 'ar-XA',
 'bg-BG',
 'bn-IN',
 'ca-ES',
 'cmn-CN',
 'cmn-TW',
 'cs-CZ',
 'da-DK',
 'de-DE',
 'el-GR',
 'en-AU',
 'en-GB',
 'en-IN',
 'en-US',
 'es-ES',
 'es-US',
 'et-EE',
 'eu-ES',
 'fi-FI',
 'fil-PH',
 'fr-CA',
 'fr-FR',
 'gl-ES',
 'gu-IN',
 'he-IL',
 'hi-IN',
 'hu-HU',
 'id-ID',
 'is-IS',
 'it-IT',
 'ja-JP',
 'kn-IN',
 'ko-KR',
 'lt-LT',
 'lv-LV',
 'ml-IN',
 'mr-IN',
 'ms-MY',
 'nb-NO',
 'nl-BE',
 'nl-NL',
 'pa-IN',
 'pl-PL',
 'pt-BR',
 'pt-PT',
 'ro-RO',
 'ru-RU',
 'sk-SK',
 'sr-RS',
 'sv-SE',
 'sw-KE',
 'ta-IN',
 'te-IN',
 'th-TH',
 'tr-TR',
 'uk-UA',
 'ur-IN',
 'vi-VN',
 'yue-HK']


In [10]:
voices_df = voices_df.explode("language_codes")
voices_df = voices_df[voices_df["language_codes"].isin(LANGUAGE_CODES)]
voices_df = voices_df.sort_values(by=["language_codes", "gender"]).reset_index(drop=True)
voices_df.head(3)

,name,language_codes,gender
0,en-GB-Chirp-HD-F,en-GB,FEMALE
1,en-GB-Chirp-HD-O,en-GB,FEMALE
2,en-GB-Chirp3-HD-Achernar,en-GB,FEMALE


We'll limit the search for HD models only, as they limit our scope to Chrisp 3: HD voices, that are, according to [Google TTS Docs](https://cloud.google.com/text-to-speech/pricing?hl=en), the latest TTS models as of 2025-10-11.

In [11]:
voices_df["model_name"] = voices_df["name"].str.split("-HD-").apply(lambda x: x[1] if len(x) > 1 else None)
voices_df.dropna(subset=["model_name"], inplace=True)

Limit models list to only those available in all languages.

In [12]:
groups = voices_df.groupby(["language_codes"])
common_model_names = set.intersection(*[set(group["model_name"].unique()) for _, group in groups])
voices_df = voices_df[voices_df["model_name"].isin(common_model_names)]
voices_df.head(3)

,name,language_codes,gender,model_name
0,en-GB-Chirp-HD-F,en-GB,FEMALE,F
1,en-GB-Chirp-HD-O,en-GB,FEMALE,O
2,en-GB-Chirp3-HD-Achernar,en-GB,FEMALE,Achernar


There are still many models to choose from. Official demo by Google uses F for Female and D for male, and that's what we'll use as well.

In [13]:
voices_df[["gender", "model_name"]].drop_duplicates().sort_values(by=["gender", "model_name"]).reset_index(drop=True)

,gender,model_name
0,FEMALE,Achernar
1,FEMALE,Aoede
2,FEMALE,Autonoe
3,FEMALE,Callirrhoe
4,FEMALE,Despina
5,FEMALE,Erinome
6,FEMALE,F
7,FEMALE,Gacrux
8,FEMALE,Kore
9,FEMALE,Laomedeia


In [14]:
voices_df[voices_df["model_name"].isin(["F", "D"])][["language_codes", "gender", "name"]].drop_duplicates().sort_values(
    by=["language_codes", "gender"]
).reset_index(drop=True).rename(columns={"language_codes": "language_code"})

,language_code,gender,name
0,en-GB,FEMALE,en-GB-Chirp-HD-F
1,en-GB,MALE,en-GB-Chirp-HD-D
2,es-ES,FEMALE,es-ES-Chirp-HD-F
3,es-ES,MALE,es-ES-Chirp-HD-D
4,fr-FR,FEMALE,fr-FR-Chirp-HD-F
5,fr-FR,MALE,fr-FR-Chirp-HD-D


## Section cleanup

In [15]:
del voices_df

# Voices demonstration

In [ ]:
dt: list[dict[str, Any]] = []

for language, gender_config in TTS_CONFIGS.items():
    for gender, config in gender_config.items():
        audio = await tts.synthesize_text(  # type: ignore[top-level-await] # pylint: disable=await-outside-async # noqa: E501,F704
            text=SAMPLES[language],
            language_code=config.language_code,
            voice_name=config.voice_name,
            gender=config.gender,  # type: ignore[arg-type]
        )

        dt.append(
            {
                "language_code": config.language_code,
                "gender": config.gender,
                "voice_name": config.voice_name,
                "audio": audio,
            }
        )

sample_df = pd.DataFrame(dt, columns=["language_code", "gender", "voice_name", "audio"])
del dt

sample_df.head(3)

,language_code,gender,voice_name,audio
0,fr-FR,1,None,b'\xff\xf3\x84\xc4\x00\x00\x00\x00\x00\x00\x00...
1,fr-FR,2,None,b'\xff\xf3\x84\xc4\x00\x00\x00\x00\x00\x00\x00...
2,en-GB,1,None,b'\xff\xf3\x84\xc4\x00\x00\x00\x00\x00\x00\x00...


In [17]:
_ = sample_df.apply(display_sample_df, axis=1)

Language: fr-FR, Gender: 1, Voice: None


Language: fr-FR, Gender: 2, Voice: None


Language: en-GB, Gender: 1, Voice: None


Language: en-GB, Gender: 2, Voice: None


Language: es-ES, Gender: 1, Voice: None


Language: es-ES, Gender: 2, Voice: None


## Section cleanup

In [18]:
del sample_df